# CCA-F Review Notebook — Prompt Engineering

This notebook is organized by the provided CCA-F task list. Each task includes a concept explanation, runnable Python example, exam summary, and two practice questions.

## Environment Setup

In [ ]:
import json
import re
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Optional

MODEL = "claude-haiku-4-5-20251001"
print("Environment ready: examples are local simulations shaped like Claude API and Claude Code workflows.")

## D4.1 — Apply few-shot techniques and structured prompt patterns

### 📖 Concept Explanation

Few-shot prompting is not just adding examples; it uses a small set of high-signal examples to define task boundaries, output shape, and failure behavior. CCA-F questions often test this judgment: when a task has ambiguous classification boundaries, extraction edge cases, or strict formatting needs, 2-4 representative positive, negative, and boundary examples are usually more effective than longer prose instructions.

Structured prompts should separate role, rules, examples, source material, and output contract. XML tags are common with Claude because sections such as `<role>`, `<rules>`, `<examples>`, `<document>`, and `<output_format>` reduce the chance that the model blends instructions with data. Role assignment should be specific about responsibility and constraints, such as “you are an invoice field extractor,” not a vague “you are smart.”

In [ ]:
# Task 4.1: few-shot examples, XML tags, and role assignment.
# This does not call a model; it shows the prompt structure and simulates the expected extraction locally.

import json
import re

prompt = """<role>You are a careful invoice field extractor. Return only JSON that matches the contract.</role>
<rules>
- Extract only fields explicitly present in <document>.
- Return null for missing fields; do not guess or complete values.
- Currency must be USD, CAD, EUR, or null.
</rules>
<examples>
<example type="positive">
  <document>Invoice INV-100. Total: $42.00 USD</document>
  <answer>{"invoice_id":"INV-100","total":42.0,"currency":"USD"}</answer>
</example>
<example type="negative">
  <document>Invoice INV-101. Amount due: not provided.</document>
  <answer>{"invoice_id":"INV-101","total":null,"currency":null}</answer>
</example>
<example type="boundary">
  <document>Quote Q-1. Estimated total may be 80 CAD.</document>
  <answer>{"invoice_id":null,"total":null,"currency":"CAD"}</answer>
</example>
</examples>
<output_format>{"invoice_id": string|null, "total": number|null, "currency": "USD"|"CAD"|"EUR"|null}</output_format>
<document>Invoice INV-9, Total: $88.10 USD</document>
"""

# In production this prompt would be sent to the model. This local function keeps the notebook runnable offline.
def extract_invoice_locally(document: str) -> dict:
    invoice_match = re.search(r"Invoice\s+(INV-\d+)", document)
    total_match = re.search(r"Total:\s*\$([0-9]+(?:\.[0-9]{2})?)", document)
    currency_match = re.search(r"\b(USD|CAD|EUR)\b", document)
    return {
        "invoice_id": invoice_match.group(1) if invoice_match else None,
        "total": float(total_match.group(1)) if total_match else None,
        "currency": currency_match.group(1) if currency_match else None,
    }

source_document = "Invoice INV-9, Total: $88.10 USD"
print("Structured prompt document section:")
print(prompt.split("<document>")[-1].strip())
print("Local simulated output:")
print(json.dumps(extract_invoice_locally(source_document), indent=2))

### ⚠️ Exam Summary & Common Traps

**✅ Correct patterns**

- Use a small set of high-quality few-shot examples covering normal, negative, and boundary cases.
- Use XML tags to separate role, rules, examples, source material, and output format.
- Make role assignment concrete: task responsibility, constraints, and expected output.
- Negative examples should show what not to extract or infer, not just show malformed text.

**✗ Anti-patterns / distractors**

- Replacing examples with long prose while leaving the boundary unclear.
- Providing only positive examples and never showing missing, ambiguous, or conflicting information.
- Asking for hidden chain-of-thought instead of concise rationale, validation status, or structured fields.
- Mixing real input into the examples section so the model treats it as another example.

### 🧪 Practice Questions

**Q1.** A Claude app classifies support tickets as `refund`, `technical_issue`, or `billing_question`. It repeatedly misclassifies “wrong invoice address” as `technical_issue`. What is the best CCA-F-style improvement?

A) Add a boundary few-shot example such as “wrong invoice address => billing_question” while keeping the enum constraint  
B) Increase temperature so the model explores more labels  
C) Remove the enum and let the model invent labels  
D) Ask the model to output its hidden chain-of-thought

> **Answer: A**  
> **Explanation:** Boundary failures are best handled with targeted examples and explicit label constraints; more randomness or open-ended labels reduce reliability.

**Q2.** Which prompt structure best reduces the risk that examples are treated as the current input?

A) Separate examples and the current source with `<examples>` and `<document>` tags  
B) Put rules, examples, and source text in one unstructured paragraph  
C) Only say “you are an expert, follow the instructions strictly”  
D) Remove negative examples so the model never sees bad cases

> **Answer: A**  
> **Explanation:** XML tags clearly isolate examples from current source material. Negative examples are useful when their boundaries are clear.

## D4.2 — Enforce structured output with JSON Schema and Pydantic

### 📖 Concept Explanation

Structured output is not the same as saying “return JSON.” The output contract should be represented in a verifiable schema or type model. JSON Schema and Pydantic-style validators can check required fields, types, enums, ranges, and cross-field business rules. A common exam trap is treating “parseable JSON” as “correct output”; valid JSON can still contain the wrong type, an invalid enum value, or a value unsupported by the source.

A production design usually has three layers: the prompt states the expected shape, the model/tool schema constrains structure, and programmatic validation checks the result. When validation fails, precise errors should be fed back to the model for self-correction. If source information is absent, the schema should allow `null` or `unknown` instead of forcing fabrication.

In [ ]:
# Task 4.2: JSON Schema / Pydantic-style validation, retry feedback, and self-correction.
# This uses only the Python standard library to simulate schema validation and correction feedback.

import json

invoice_schema = {
    "type": "object",
    "required": ["invoice_id", "total", "currency"],
    "properties": {
        "invoice_id": {"type": ["string", "null"]},
        "total": {"type": ["number", "null"], "minimum": 0},
        "currency": {"type": "string", "enum": ["USD", "CAD", "EUR", "unknown"]},
    },
}


def validate_invoice(data: dict) -> list[str]:
    errors = []
    required = invoice_schema["required"]
    properties = invoice_schema["properties"]

    for field in required:
        if field not in data:
            errors.append(f"missing required field: {field}")

    invoice_id = data.get("invoice_id")
    if invoice_id is not None and not isinstance(invoice_id, str):
        errors.append("invoice_id must be string or null")

    total = data.get("total")
    if total is not None:
        if not isinstance(total, (int, float)) or isinstance(total, bool):
            errors.append("total must be number or null")
        elif total < properties["total"]["minimum"]:
            errors.append("total cannot be negative")

    if data.get("currency") not in properties["currency"]["enum"]:
        errors.append("currency must be USD, CAD, EUR, or unknown")

    return errors


def self_correct(candidate: dict, errors: list[str]) -> dict:
    # A real system would send errors, source text, and failed JSON back to the model.
    # This deterministic function simulates the correction step.
    corrected = dict(candidate)
    if "total must be number or null" in errors and isinstance(corrected.get("total"), str):
        corrected["total"] = float(corrected["total"])
    if "currency must be USD, CAD, EUR, or unknown" in errors:
        corrected["currency"] = "unknown"
    return corrected

candidate = {"invoice_id": "INV-9", "total": "88.10", "currency": "dollars"}
errors = validate_invoice(candidate)
print("First validation errors:", errors)
retry_feedback = {"failed_output": candidate, "validation_errors": errors}
print("Retry feedback sent to the model:")
print(json.dumps(retry_feedback, indent=2))

corrected = self_correct(candidate, errors)
print("Corrected output:", corrected)
print("Second validation errors:", validate_invoice(corrected))

### ⚠️ Exam Summary & Common Traps

**✅ Correct patterns**

- Use JSON Schema, tool schemas, or Pydantic-style models for fields, types, enums, and ranges.
- Allow `null` or `unknown` for fields that may be absent in the source material.
- Feed precise validation errors back to retries, such as missing fields, type errors, invalid enums, or negative amounts.
- Distinguish syntax validation, schema validation, and business semantic validation.

**✗ Anti-patterns / distractors**

- Only writing “must return JSON” in the prompt with no programmatic validation.
- Making every field non-null and accidentally encouraging fabricated values.
- Treating successful `json.loads()` as proof that the answer is trustworthy.
- Retrying with vague feedback such as “fix the format” instead of concrete validation errors.

### 🧪 Practice Questions

**Q1.** The model returns `{"total":"88.10","currency":"dollars"}`, but the schema requires `total` as a number and `currency` as `USD|CAD|EUR|unknown`. What is the best next step?

A) Feed field-level validation errors back to the model and ask it to correct only the invalid fields  
B) Accept the result because it is valid JSON  
C) Remove the enum constraint to avoid failures  
D) Retry forever until the model randomly returns the right shape

> **Answer: A**  
> **Explanation:** Valid JSON is not necessarily schema-valid. Specific feedback supports controlled self-correction.

**Q2.** The source document does not include a supplier tax ID, but the output schema contains `supplier_tax_id`. Which schema design is most robust?

A) Allow `supplier_tax_id` to be `string|null` and instruct the model to return `null` when absent  
B) Require `supplier_tax_id` as a non-empty string  
C) Ask the model to infer the tax ID from the company name  
D) Remove the field from validation but still ask the model to output it

> **Answer: A**  
> **Explanation:** Nullable fields make “unknown” a valid state and reduce fabrication pressure.

## D4.3 — Design validation retry loops for output reliability

### 📖 Concept Explanation

A validation retry loop wraps probabilistic generation in deterministic control flow: generate a candidate, validate it, feed specific errors back to the model, retry, and then escalate or fall back after a bounded number of attempts. CCA-F focuses on bounded reliability: retries can fix format, type, and enum errors, but they cannot create evidence that is absent from the source material.

A strong retry loop has a maximum retry threshold, observable error history, and a clear escalation path. The retry prompt should not merely say “try again”; it should include the original source, failed output, schema or business errors, and the allowed correction scope.

In [ ]:
# Task 4.3: validation retry loop, maximum threshold, and fallback escalation.
# The key pattern is specific feedback, bounded retries, and a recoverable fallback.

from dataclasses import dataclass

@dataclass
class ExtractionAttempt:
    attempt: int
    output: dict
    errors: list[str]


def simulated_model(attempt: int, feedback: dict | None) -> dict:
    # The first attempt has a type error; specific feedback lets the second attempt correct it.
    if attempt == 1:
        return {"invoice_id": "INV-9", "total": "88.10", "currency": "USD"}
    if feedback and "total must be number or null" in feedback.get("validation_errors", []):
        return {"invoice_id": "INV-9", "total": 88.10, "currency": "USD"}
    return {"invoice_id": "INV-9", "total": None, "currency": "unknown"}


def validate_extraction(data: dict) -> list[str]:
    errors = []
    if not isinstance(data.get("invoice_id"), str):
        errors.append("invoice_id must be string")
    if data.get("total") is not None and not isinstance(data.get("total"), (int, float)):
        errors.append("total must be number or null")
    if data.get("currency") not in {"USD", "CAD", "EUR", "unknown"}:
        errors.append("currency enum is invalid")
    return errors


def extract_with_retries(max_retries: int = 2) -> dict:
    feedback = None
    history: list[ExtractionAttempt] = []

    for attempt in range(1, max_retries + 2):
        output = simulated_model(attempt, feedback)
        errors = validate_extraction(output)
        history.append(ExtractionAttempt(attempt, output, errors))
        print(f"attempt={attempt}", output, "errors=", errors)

        if not errors:
            return {"status": "accepted", "result": output, "attempts": attempt}

        feedback = {"failed_output": output, "validation_errors": errors}

    # Stop at the threshold and use a fallback instead of retrying forever.
    return {
        "status": "needs_human_review",
        "result": None,
        "last_errors": history[-1].errors,
        "attempts": len(history),
    }

final_result = extract_with_retries(max_retries=2)
print("final:", final_result)

### ⚠️ Exam Summary & Common Traps

**✅ Correct patterns**

- Include source text, failed output, and specific validation errors in each retry.
- Set a maximum retry count and log each failure reason for debugging and auditability.
- Retry correctable errors; escalate or return `null` when evidence is missing or risk is high.
- Fallbacks can include human review, conservative defaults, queued reprocessing, or requesting better upstream data.

**✗ Anti-patterns / distractors**

- Infinite retries that let latency, cost, and uncertainty grow without bounds.
- Retry prompts that only say “try again” with no error localization.
- Treating every validation failure as a prompt problem without distinguishing schema errors, business errors, and missing evidence.
- Silently returning the last partial or invalid result after the threshold is reached.

### 🧪 Practice Questions

**Q1.** An extraction flow fails twice: first `total` is a string, then `currency` is outside the enum. What is the best retry-loop design?

A) Feed each specific validation error, failed output, and source text back to the model, then escalate after the maximum attempts  
B) Keep calling the model without recording errors until it succeeds  
C) Remove validation so the workflow can continue  
D) Only increase max tokens

> **Answer: A**  
> **Explanation:** Bounded retries need specific feedback and a maximum threshold. After that threshold, the system needs an explicit fallback.

**Q2.** The maximum retry count is reached and the required evidence still cannot be found in the document. Which action best matches CCA-F guidance?

A) Return `needs_human_review` or `null`, preserving the last validation errors  
B) Ask the model to fill the field from general knowledge  
C) Continue retrying forever because the next call might work  
D) Swallow the error and return the last model output

> **Answer: A**  
> **Explanation:** Missing source evidence is not solved by more retries; reliable systems need stop conditions, fallbacks, and observable errors.